# EuroSAT Land-Cover Classifier

An original compact CNN project for 10-class land-cover classification from Sentinel-2 RGB patches. It downloads and checksum-verifies the public EuroSAT archive, creates reproducible stratified train/validation/test partitions, trains a model from scratch, reports held-out metrics, and creates an explainability visualization.

Base paper: Helber et al. (2019), [EuroSAT](https://arxiv.org/abs/1709.00029). Dataset record: [Zenodo DOI 10.5281/zenodo.7711810](https://zenodo.org/records/7711810). The code here is independently implemented; see `SOURCES.md` for the dataset citation and workflow reference.


## 1. Install dependencies
If using Colab, select a GPU runtime first. CPU is supported but takes longer. In VS Code, run this notebook from the `EuroSAT_Project` folder.


In [ ]:
%pip install -q -r requirements.txt


In [ ]:
import torch, platform
print('Python:', platform.python_version())
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())


## 2. Download and verify EuroSAT RGB
The public RGB archive is about 95 MB. The downloader verifies the Zenodo MD5 before extraction. The dataset is stored under `data/` and remains separate from the source code.


In [ ]:
%run download_data.py


## 3. Train and evaluate
The script writes a fixed-seed stratified split manifest, checkpoint, training history, test metrics, class report, confusion matrix, and learning curve into `outputs/`. `--epochs` can be reduced for a quick smoke run; record the chosen setting in any report.


In [ ]:
EPOCHS = 8
!python train.py --data data/EuroSAT_RGB --epochs {EPOCHS} --batch-size 64


In [ ]:
import json
from pathlib import Path
metrics = json.loads(Path('outputs/metrics.json').read_text())
{k: metrics[k] for k in ['test_accuracy','test_macro_f1','test_samples','best_epoch','device']}


In [ ]:
from IPython.display import display, Image
display(Image(filename='outputs/confusion_matrix.png'))
display(Image(filename='outputs/training_curve.png'))


## 4. Try a held-out sample and inspect Grad-CAM
This predicts a real test image from the dataset and saves a heatmap over the input. The heatmap is an attention-style diagnostic, not a causal explanation.


In [ ]:
import csv
rows = list(csv.DictReader(open('outputs/split_manifest.csv', newline='', encoding='utf-8')))
sample = next(r['path'] for r in rows if r['split']=='test')
!python predict.py data/EuroSAT_RGB/{sample} --checkpoint outputs/best_model.pt
display(Image(filename='outputs/prediction_gradcam.png'))


## Interpretation and limitations
This model uses RGB only and is trained from scratch; it does not implement the paper's full 13-band benchmark. The held-out test score is a result of this particular run and split, and should be reported with the seed, epoch count, and per-class results. Geographic leakage is possible in a random image-level split, so it does not establish geographic generalization. See `README.md` and `SOURCES.md` before reuse or redistribution.
